In [ ]:
import numpy as np
import torch
from tqdm import tqdm
from sklearn.decomposition import PCA
from nflows.flows import Flow
from nflows.distributions import StandardNormal
from nflows.transforms import CompositeTransform
from nflows.transforms.autoregressive import MaskedAffineAutoregressiveTransform


def pca_normalizing_flow(nz_ensemble, n_components=4, n_layers=5, hidden_features=32, lr=1e-3, steps=3000, batch_size=128, val_fraction=0):
    # Each row is one plausible n(z), evaluated on the same redshift grid
    nz_ensemble = np.asarray(nz_ensemble)

    # PCA finds the main ways in which the full n(z) curves vary across the ensemble
    pca = PCA(n_components=n_components)
    A = pca.fit_transform(nz_ensemble)  # A contains the PCA coordinates of every reconstruction

    # Standardise the PCA coordinates so that the NF sees parameters with comparable scales
    mean = A.mean(axis=0)
    std = A.std(axis=0) + 1e-6
    A_norm = (A - mean) / std

    N = len(A_norm)
    idx = np.random.permutation(N)
    n_val = int(val_fraction * N)
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]

    train_data = torch.tensor(A_norm[train_idx], dtype=torch.float32)
    val_data = torch.tensor(A_norm[val_idx], dtype=torch.float32) if n_val > 0 else None

    n_params = n_components
    base = StandardNormal(shape=[n_params])  # The flow starts from a simple Gaussian and learns to transform it into the PCA distribution

    transforms = []
    for _ in range(n_layers):
        transforms.append(MaskedAffineAutoregressiveTransform(features=n_params, hidden_features=hidden_features))

    transform = CompositeTransform(transforms)  # Apply all learned transformations in sequence
    flow = Flow(transform, base)

    optimizer = torch.optim.Adam(flow.parameters(), lr=lr)
    train_losses = []
    val_losses = []

    pbar = tqdm(range(steps), desc="Training PCA + NF", ncols=100)
    for step in pbar:
        batch_idx = torch.randint(0, train_data.shape[0], (min(batch_size, train_data.shape[0]),))
        batch = train_data[batch_idx]

        train_loss = -flow.log_prob(batch).mean()  # Maximum likelihood training: make plausible PCA coordinates have high probability
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()
        train_losses.append(train_loss.item())

        if val_data is not None:
            with torch.no_grad():
                val_loss = -flow.log_prob(val_data).mean()
            val_losses.append(val_loss.item())

    return flow, pca, mean, std, train_losses, val_losses


def evaluate_pca_nf_prior(nz_current, flow, pca, mean, std):
    nz_current = np.asarray(nz_current)

    if nz_current.ndim == 1:
        nz_current = nz_current[None, :]  # PCA expects a 2D array, even for one reconstruction

    A_current = pca.transform(nz_current)  # Express the current n(z) in the same coordinates learned from the training ensemble
    A_current_norm = (A_current - mean) / std  # Apply exactly the same standardisation used during training

    A_current_tensor = torch.tensor(A_current_norm, dtype=torch.float32)

    with torch.no_grad():
        log_prob = flow.log_prob(A_current_tensor)  # Joint log probability of the complete reconstruction in PCA space

    return log_prob.item()



In [ ]:
flows = []
pcas = []
means = []
stds = []
train_losses = []
val_losses = []

for ensemble in nz_ensemble_list:
    flow, pca, mean, std, train_loss, val_loss = pca_normalizing_flow(
        ensemble,
        n_components=4,
        n_layers=5,
        hidden_features=32,
        lr=1e-3,
        steps=3000,
        batch_size=128,
        val_fraction=0
    )

    flows.append(flow)
    pcas.append(pca)
    means.append(mean)
    stds.append(std)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

In [ ]:
def sample_nz_from_flow(flow, pca, mean, std, n_samples=100):
    with torch.no_grad():
        samples_norm = flow.sample(n_samples).cpu().numpy()
    samples = samples_norm * std + mean
    nz_samples = pca.inverse_transform(samples)
    nz_samples = np.clip(nz_samples, 0, None)
    return nz_samples

In [ ]:
def plot_all_bins_pca(flows, means, stds, pcas, nz_ensemble_list, z_ens_list):
    fig, axes = plt.subplots(len(flows), 2, figsize=(22, 4*len(flows)))
    for b in range(len(flows)):
        print(f"Plotting bin {b+1}")
        flow = flows[b]
        mean = means[b]
        std = stds[b]
        pca = pcas[b]
        nz_ens = nz_ensemble_list[b]
        z_grid = z_ens_list[b]
        ax_left = axes[b, 0]
        ax_right = axes[b, 1]
        U_flow_norm = flow.sample(200).detach().numpy()
        PCA_flow = U_flow_norm * std + mean
        nz_flow = pca.inverse_transform(PCA_flow)
        nz_flow = np.clip(nz_flow, 0, None)
        for i in range(len(nz_flow)):
            ax_left.plot(z_grid, nz_flow[i], color="crimson", alpha=0.2)
        ax_left.set_xlim(x_bounds[b][0], x_bounds[b][1])
        ax_left.set_title(f"Bin {b+1}: PCA + Normalising Flow Prior Samples")
        ax_left.set_xlabel("$z$")
        ax_left.set_ylabel("$n(z)$")
        ax_left.set_ylim(0, np.max(nz_flow + 1))
        ax_left.legend(["PCA + NF reconstructions"])
        for i in range(nz_ens.shape[0]):
            ax_right.plot(z_grid, nz_ens[i], color="crimson", alpha=0.2, lw=0.8)
        ax_right.set_xlim(x_bounds[b][0], x_bounds[b][1])
        ax_right.set_title(f"Bin {b+1}: Ensemble Curves")
        ax_right.set_xlabel("$z$")
        ax_right.set_ylabel("$n(z)$")
        ax_right.set_ylim(0, np.max(nz_flow + 1))
    plt.tight_layout()
    return fig

In [ ]:
fig = plot_all_bins_pca(flows, means, stds, pcas, nz_ensemble_list, z_ens_list)

In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(16, 18))

for b in range(6):
    pca_samples = pcas[b].transform(nz_ensemble_list[b])
    flow_samples_norm = flows[b].sample(1000).detach().numpy()
    flow_samples = flow_samples_norm * stds[b] + means[b]

    for i in range(4):
        bins = np.histogram_bin_edges(np.concatenate([pca_samples[:, i], flow_samples[:, i]]),bins=20)
        axes[b, i].hist(pca_samples[:, i], bins=bins, alpha=0.4, label="Ensemble",  density=True)
        axes[b, i].hist(flow_samples[:, i], bins=bins, histtype="step", linewidth=1.8, label="NF samples", density=True)
        axes[b, i].set_xlabel(fr"PC$_{i+1}$")
    axes[b, 0].set_ylabel(f"Bin {b+1}")
for i in range(4):
    axes[0, i].set_title(fr"PC$_{i+1}$")

axes[0, 0].legend()

plt.tight_layout()